# Entrenamiento v2 — detección de daños en palas (OV9281 + Coral)

Esta versión aplica los cambios del informe de mejoras:

1. **Dos datasets**: DTU/Nordtank (se reconstruye, re-muestrea y corta en tiles) + WTBs2025_2cls (solo gris + ruido, sin re-muestrear ni cortar). `train` y `val` usan los dos; el **test de decisión es solo DTU**. El test de WTBs se reporta aparte, con recall por clase original.
2. **Orden de clases `0 = damage`, `1 = dirt`** (el DTU original viene al revés y se re-mapea al leerlo).
3. **Split 70/15/15 por foto original** (no por tile), con set de test separado.
4. **Reconstruye cada foto** a partir de sus tiles 586×371 y une los daños cortados en el borde de un tile.
5. **Simula la cámara OV9281**: re-muestrea al GSD objetivo (5 mm/px por defecto), pasa a escala de grises y agrega ruido y blur de sensor.
6. **Corta tiles de 640×400** (la mitad del frame de 1280×800) y suma **negativos** (pala limpia, solo DTU) hasta el 20 % de train; en val y test se conservan todos los tiles limpios. Los tiles donde se ve un daño que no se pudo etiquetar (muy chico o muy cortado) se descartan: no son pala limpia.
7. **Entrenamiento más largo** (200 epochs, patience 50), sin aumentos de color, semillas fijas.
8. **Evaluación orientada a inspección**: el umbral de confianza se elige en **val** (parte DTU) y se aplica fijo en test; recall de *damage* por tamaño (asignación 1 a 1) y falsos positivos cada 100 tiles limpios.
9. **Export a Edge TPU calibrado con una muestra aleatoria propia** (~300 imágenes de val) y validación del modelo int8.

**Cómo correrlo en Kaggle**

- *Add Input* → dataset `ajifoster3/yolo-annotated-wind-turbines-586x371`.
- *Add Input* → dataset privado con `WTBs2025_2cls` (subir a Kaggle el zip de la carpeta de Drive). El notebook lo busca solo en `/kaggle/input` (una carpeta con "wtbs" en el nombre y `images/` + `labels/` por split, o `train/images` …); si no lo encuentra se puede fijar `WTBS_DIR`. Con `USAR_WTBS = False` se entrena solo con DTU (para comparar).
- *Settings* → Accelerator **GPU T4 ×2** (o P100) e **Internet on**.
- Ajustar la celda de configuración y *Run All*. Al final queda `/kaggle/working/entregables.zip` con el modelo, el `.tflite` para la Coral y las métricas.

Para el experimento de escala (E3) se corre tres veces cambiando `GSD_OBJETIVO_MM` a 2.5, 5 y 10. Para la comparación color/gris (rama A/B) se cambia `MODO_COLOR`.

In [ ]:
# ============================================================
# 1. Instalación y configuración
# ============================================================
# Versión fija de Ultralytics: la misma del run anterior, para poder repetir resultados.
# (Se quitó el "pip install numpy<2": Ultralytics lo reinstalaba igual y dejaba conflictos.)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171"], check=False)

import os, re, csv, json, glob, math, random, shutil, collections, time
from pathlib import Path
import cv2
import numpy as np
import yaml

# ---------- Datasets ----------
USAR_WTBS = True         # True: train/val = DTU + WTBs2025_2cls. False: solo DTU (comparación)
WTBS_DIR = "/kaggle/input/datasets/juliangiles/dataset-wtg"          # ruta a la carpeta WTBs2025_2cls en /kaggle/input (None = se busca sola)
WTBS_COL_IMG = None      # columnas de manifest.csv con la imagen y la clase original (None = se detectan)
WTBS_COL_CLASE = None
VARIANTES_WTBS = 1       # versiones de cada imagen WTBs en train (ya son muchas imágenes y muy variadas)

# ---------- Datos y simulación de la cámara ----------
SEED = 42
GSD_ORIG_MM = 1.5        # mm/px estimado de las fotos de detalle del dataset (Zenmuse X5S, f=15 mm, a 5-8 m)
GSD_OBJETIVO_MM = 5.0    # mm/px del OV9281: lente 3 mm a 5 m. Experimento E3: probar 2.5 y 10
MODO_COLOR = "gris"      # "gris" = OV9281 (rama B) | "rgb" = referencia color (rama A)
SOLO_DANIO = False       # False: entrena damage + dirt (dirt enseña qué NO es daño). True: solo damage
TILE_W, TILE_H = 640, 400   # mitad del frame 1280x800 del OV9281 (tiling 2x2)
SOLAPE = 0.15
VARIANTES_TRAIN = 3      # versiones distintas (escala, ruido, gris) de cada foto DTU de train
NEG_FRAC = 0.20          # fracción máxima de tiles sin cajas...
NEG_LIMITAR = ("train",) # ...solo en train. En val y test se dejan todos (FP/100 limpios más estable)
MIN_BOX_PX = 4           # cajas con lado mayor < 4 px se descartan (indetectables)
MIN_VISIBLE = 0.40       # una caja cortada por el tile se conserva si queda >= 40 % visible
MIN_VALIDO = 0.50        # un tile NEGATIVO debe tener >= 50 % de imagen real (el resto es relleno)
SPLIT = (0.70, 0.15, 0.15)
ORIGINALES_DIR = None    # opcional: carpeta con las fotos originales de DTU (DJI_XXXX.JPG) en /kaggle/input

# ---------- Entrenamiento ----------
MODELO_BASE = "yolov8n.pt"   # experimento E5: "yolo26n.pt"
EPOCHS, PATIENCE, BATCH, IMGSZ = 200, 50, 32, 640
FRACTION = 1.0               # 0.25 + EPOCHS=60 para iterar rápido
P_MIN = 0.60                 # precisión mínima al elegir el umbral de confianza (en val)
N_CALIB = 300                # imágenes de val (al azar) para calibrar la cuantización int8

WORK = Path("/kaggle/working")
TAG = "dtu+wtbs" if USAR_WTBS else "dtu"
OUT_DS = WORK / f"ds_{TAG}_{MODO_COLOR}_{GSD_OBJETIVO_MM:g}mm"
RUN_NAME = f"{Path(MODELO_BASE).stem}_{TAG}_{MODO_COLOR}_{GSD_OBJETIVO_MM:g}mm"

random.seed(SEED)
np.random.seed(SEED)
print("Corrida:", RUN_NAME)
print(f"Factor de escala: {min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM):.3f}  "
      f"(un daño de 29 px pasa a ~{29 * min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM):.0f} px)")

In [ ]:
# ============================================================
# 2. Indexar los datasets: DTU (tiles -> fotos originales) y WTBs2025_2cls
# ============================================================
def encontrar_dataset():
    candidatos = [Path("/kaggle/input/datasets/ajifoster3/yolo-annotated-wind-turbines-586x371/NordTank586x371")]
    candidatos += [Path(p).parent for p in glob.glob("/kaggle/input/**/labels", recursive=True)]
    for c in candidatos:
        if "wtbs" in str(c).lower():          # no confundir con el segundo dataset
            continue
        if (c / "images").is_dir() and (c / "labels").is_dir():
            return c
    raise FileNotFoundError("No encontré images/ y labels/ en /kaggle/input. ¿Agregaste el dataset al notebook?")

DATASET = encontrar_dataset()
IMG_DIR, LBL_DIR = DATASET / "images", DATASET / "labels"
print("Dataset DTU:", DATASET)

PATRON = re.compile(r"^(DJI_\d+)_(\d+)_(\d+)$")   # DJI_0201_02_06 -> foto, índice A, índice B
imgs, sin_patron = {}, []
for p in IMG_DIR.iterdir():
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        if PATRON.match(p.stem):
            imgs[p.stem] = p
        else:
            sin_patron.append(p.name)
print(f"Tiles: {len(imgs)}  (sin formato DJI_XXXX_A_B, ignorados: {len(sin_patron)})")

A = [int(PATRON.match(s).group(2)) for s in imgs]
B = [int(PATRON.match(s).group(3)) for s in imgs]
# La foto 5280x2970 se cortó en 9 columnas x 8 filas: el índice con más valores es la columna.
if (max(A) - min(A)) >= (max(B) - min(B)):
    G_COL, G_FIL, COL0, FIL0 = 2, 3, min(A), min(B)
else:
    G_COL, G_FIL, COL0, FIL0 = 3, 2, min(B), min(A)
print(f"Índice A: {min(A)}..{max(A)}  Índice B: {min(B)}..{max(B)}  ->  columna = grupo {G_COL - 1}")

muestra = cv2.imread(str(next(iter(imgs.values()))))
TH0, TW0 = muestra.shape[:2]            # 371, 586
fotos = collections.defaultdict(list)    # foto -> [(stem, fila, col)]
for s in imgs:
    m = PATRON.match(s)
    fotos[m.group(1)].append((s, int(m.group(G_FIL)) - FIL0, int(m.group(G_COL)) - COL0))
NCOL = max(c for v in fotos.values() for _, _, c in v) + 1
NFIL = max(f for v in fotos.values() for _, f, _ in v) + 1
print(f"Fotos: {len(fotos)}  grilla {NCOL}x{NFIL} tiles de {TW0}x{TH0}  ->  lienzo {NCOL * TW0}x{NFIL * TH0}")

# En las etiquetas originales del DTU 0 = dirt y 1 = damage. Se pasan al orden final (0 = damage, 1 = dirt),
# el mismo de WTBs2025_2cls, para que las dos fuentes no queden con las clases cruzadas.
DTU_A_FINAL = {0: 1, 1: 0}

def leer_labels(stem):
    f = LBL_DIR / f"{stem}.txt"
    if not f.exists():
        return None
    out = []
    for line in f.read_text().splitlines():
        v = line.split()
        if len(v) >= 5:
            out.append((DTU_A_FINAL[int(v[0])],) + tuple(map(float, v[1:5])))
    return out

con_danio = {f for f, v in fotos.items() if any(leer_labels(s) for s, _, _ in v)}
print(f"Fotos con al menos una etiqueta: {len(con_danio)}  /  solo pala limpia: {len(fotos) - len(con_danio)}")

# ---------- WTBs2025_2cls ----------
EXT_IMG = {".png", ".jpg", ".jpeg", ".bmp"}

def estructura_yolo(raiz):
    """{split: (carpeta de imágenes, carpeta de labels)} si raiz tiene train/val/test en formato YOLO."""
    out = {}
    for sp in ("train", "val", "test"):
        for im, lb in ((raiz / "images" / sp, raiz / "labels" / sp), (raiz / sp / "images", raiz / sp / "labels")):
            if im.is_dir() and lb.is_dir():
                out[sp] = (im, lb)
                break
    return out if len(out) == 3 else None

def encontrar_wtbs():
    if WTBS_DIR:
        cands = [Path(WTBS_DIR)]
        cands += sorted({Path(p).parent for p in glob.glob(f"{WTBS_DIR}/**/labels", recursive=True)} | {Path(p).parent.parent for p in glob.glob(f"{WTBS_DIR}/**/train/labels", recursive=True)})
    else:
        cands = {Path(p).parent for p in glob.glob("/kaggle/input/**/labels", recursive=True)}
        cands |= {Path(p).parent.parent for p in glob.glob("/kaggle/input/**/train/labels", recursive=True)}
        cands = sorted(c for c in cands if "wtbs" in str(c).lower())
    for c in cands:
        e = estructura_yolo(c)
        if e:
            return c, e
    return None, None

def leer_manifest(raiz, stems):
    """Clase original de cada imagen según manifest.csv -> {stem: clase}. {} si no se puede leer."""
    rutas = sorted(set(raiz.rglob("manifest.csv")) | set(raiz.parent.glob("manifest.csv")))
    if not rutas:
        print("  manifest.csv no encontrado: no habrá recall por clase original.")
        return {}
    with open(rutas[0], newline="", encoding="utf-8-sig") as fh:
        filas = list(csv.DictReader(fh))
    if not filas:
        return {}
    cols = list(filas[0].keys())
    col_img = WTBS_COL_IMG or max(cols, key=lambda c: sum(Path(str(f[c])).stem in stems for f in filas))
    if WTBS_COL_CLASE:
        col_cls = WTBS_COL_CLASE
    else:
        claves = ("orig", "clase", "class", "categ", "defect", "tipo")
        cand = sorted((c for c in cols if c != col_img and any(k in c.lower() for k in claves)),
                      key=lambda c: 0 if "orig" in c.lower() else 1)
        col_cls = cand[0] if cand else None
    print(f"  manifest: {rutas[0]}  columnas {cols}  ->  imagen = '{col_img}', clase original = '{col_cls}'")
    if col_cls is None:
        print("  No reconocí la columna de clase: fijar WTBS_COL_CLASE en la configuración.")
        return {}
    return {Path(str(f[col_img])).stem: str(f[col_cls]) for f in filas}

WTBS, WTBS_RAIZ, WTBS_CLASE_ORIG = None, None, {}
if USAR_WTBS:
    WTBS_RAIZ, WTBS = encontrar_wtbs()
    if not WTBS:
        raise FileNotFoundError("USAR_WTBS = True pero no encontré WTBs2025_2cls en /kaggle/input. Agregar el "
                                "dataset, fijar WTBS_DIR, o poner USAR_WTBS = False para entrenar solo con DTU.")
    n = {sp: sum(p.suffix.lower() in EXT_IMG for p in d[0].iterdir()) for sp, d in WTBS.items()}
    print(f"\nDataset WTBs2025: {WTBS_RAIZ}  imágenes {n}")
    stems = {p.stem for d in WTBS.values() for p in d[0].iterdir()}
    WTBS_CLASE_ORIG = leer_manifest(WTBS_RAIZ, stems)
    if WTBS_CLASE_ORIG:
        print("  clases originales:", dict(collections.Counter(WTBS_CLASE_ORIG.values())))

In [ ]:
# ============================================================
# 3. Split 70/15/15 por foto (evita la fuga entre tiles de una misma foto)
# ============================================================
def split_por_foto(lista, seed=SEED, p=SPLIT):
    lista = sorted(lista)
    random.Random(seed).shuffle(lista)
    a, b = round(p[0] * len(lista)), round((p[0] + p[1]) * len(lista))
    return lista[:a], lista[a:b], lista[b:]

# Se divide por separado fotos con y sin daño para que los tres splits tengan de ambas
tr1, va1, te1 = split_por_foto(con_danio)
tr2, va2, te2 = split_por_foto(set(fotos) - con_danio)
SPLITS = {"train": tr1 + tr2, "val": va1 + va2, "test": te1 + te2}
for k, v in SPLITS.items():
    print(f"{k:5s}: {len(v):4d} fotos")
assert not (set(SPLITS["train"]) & set(SPLITS["test"])), "Hay fotos repetidas entre train y test"

In [ ]:
# ============================================================
# 4. Funciones: reconstruir foto, simular OV9281, cortar tiles
# ============================================================
import zlib

NOMBRES = {0: "damage"} if SOLO_DANIO else {0: "damage", 1: "dirt"}
DANIO_ID = 0   # damage es siempre la clase 0 (DTU se re-mapea en leer_labels; WTBs2025_2cls ya viene así)

def unir_cortes(cajas, tol=3):
    """Une cajas de la misma clase que quedaron partidas por el borde entre dos tiles."""
    cajas = [list(c) for c in cajas]
    def toca(p, q):
        for a, b in ((p, q), (q, p)):
            sx = round(a[3] / TW0) * TW0             # costura vertical
            if sx > 0 and abs(a[3] - sx) <= tol and abs(b[1] - sx) <= tol and min(a[4], b[4]) > max(a[2], b[2]):
                return True
            sy = round(a[4] / TH0) * TH0             # costura horizontal
            if sy > 0 and abs(a[4] - sy) <= tol and abs(b[2] - sy) <= tol and min(a[3], b[3]) > max(a[1], b[1]):
                return True
        return False
    cambio = True
    while cambio:
        cambio = False
        for i in range(len(cajas)):
            for j in range(i + 1, len(cajas)):
                a, b = cajas[i], cajas[j]
                if a[0] == b[0] and toca(a, b):
                    cajas[i] = [a[0], min(a[1], b[1]), min(a[2], b[2]), max(a[3], b[3]), max(a[4], b[4])]
                    cajas.pop(j)
                    cambio = True
                    break
            if cambio:
                break
    return cajas

def armar_foto(foto):
    """Lienzo con los tiles en su lugar, máscara de zonas reales y cajas en píxeles de la foto."""
    W, H = NCOL * TW0, NFIL * TH0
    lienzo = np.zeros((H, W, 3), np.uint8)
    valido = np.zeros((H, W), np.uint8)
    cajas = []
    for stem, f, c in fotos[foto]:
        im = cv2.imread(str(imgs[stem]))
        x0, y0 = c * TW0, f * TH0
        h, w = min(im.shape[0], H - y0), min(im.shape[1], W - x0)
        lienzo[y0:y0 + h, x0:x0 + w] = im[:h, :w]
        valido[y0:y0 + h, x0:x0 + w] = 1
        for k, cx, cy, bw, bh in leer_labels(stem) or []:
            cajas.append([k, x0 + (cx - bw / 2) * im.shape[1], y0 + (cy - bh / 2) * im.shape[0],
                          x0 + (cx + bw / 2) * im.shape[1], y0 + (cy + bh / 2) * im.shape[0]])
    if ORIGINALES_DIR:                                # foto original completa, si está disponible
        cand = list(Path(ORIGINALES_DIR).glob(f"{foto}.*"))
        if cand:
            orig = cv2.imread(str(cand[0]))
            if orig is not None and orig.shape[0] >= H and orig.shape[1] >= W:
                lienzo, valido[:] = orig[:H, :W].copy(), 1
    if valido.min() == 0:                             # zonas sin tile (cielo / suelo): relleno suave
        w8, h8 = W // 8, H // 8                       # inpainting a baja resolución + blur, sin bordes duros
        peq = cv2.resize(lienzo, (w8, h8), interpolation=cv2.INTER_AREA)
        hueco = cv2.resize((valido == 0).astype(np.uint8), (w8, h8), interpolation=cv2.INTER_NEAREST)
        rell = cv2.inpaint(peq, hueco, 7, cv2.INPAINT_TELEA)
        rell = cv2.GaussianBlur(cv2.resize(rell, (W, H), interpolation=cv2.INTER_LINEAR), (0, 0), 8)
        lienzo[valido == 0] = rell[valido == 0]
    return lienzo, valido, unir_cortes(cajas)

def a_gris(img, rng, aleatorio):
    """Escala de grises. En train los pesos B,G,R varían: el OV9281 tiene otra respuesta espectral (y ve IR)."""
    w = rng.dirichlet([3, 6, 2]) if aleatorio else np.array([0.114, 0.587, 0.299])
    return (img.astype(np.float32) * w.astype(np.float32)).sum(axis=2)

def degradar(f, rng, fuerte):
    """f: float32 0-255 (HxW o HxWx3). Simula lente + sensor. fuerte=True en train, suave y fijo en val/test."""
    h, w = f.shape[:2]
    f = cv2.GaussianBlur(f, (0, 0), rng.uniform(0.5, 1.2) if fuerte else 0.7)
    if fuerte and rng.random() < 0.3:                 # vibración: blur de movimiento leve
        k = int(rng.integers(3, 6))
        ker = np.zeros((k, k), np.float32)
        ker[k // 2, :] = 1
        M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
        ker = cv2.warpAffine(ker, M, (k, k))
        f = cv2.filter2D(f, -1, ker / max(ker.sum(), 1e-6))
    if fuerte:                                        # exposición distinta
        f = 255.0 * (np.clip(f, 0, 255) / 255.0) ** rng.uniform(0.7, 1.4)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)    # viñeteo de lente gran angular
    r2 = ((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2
    vig = 1 - (rng.uniform(0.05, 0.25) if fuerte else 0.12) * r2 / 2
    f = f * (vig[..., None] if f.ndim == 3 else vig)
    lectura = rng.uniform(2, 6) if fuerte else 3.0    # ruido de disparo + ruido de lectura
    ruido = rng.standard_normal(f.shape).astype(np.float32) * np.sqrt(np.clip(f, 0, 255) * 0.05 + lectura ** 2)
    return np.clip(f + ruido, 0, 255).astype(np.uint8)

def posiciones(largo, tile, solape=SOLAPE):
    if largo <= tile:
        return [0]
    paso = int(tile * (1 - solape))
    return sorted(set(list(range(0, largo - tile, paso)) + [largo - tile]))

def generar_tiles(lienzo, valido, cajas, split, variante, rng, cont=None):
    """Devuelve [(tile uint8, [(clase, x1, y1, x2, y2)] en px del tile, x0, y0)] al GSD objetivo.

    - Un tile con cajas válidas se conserva siempre, aunque tenga mucho relleno (la pala suele ser una
      franja angosta y los recortes sin pala del DTU se quitaron: MIN_VALIDO solo se aplica a negativos).
    - Un tile sin cajas válidas en el que SÍ se ve un daño (demasiado chico o demasiado cortado para
      etiquetarlo) no se devuelve: no es pala limpia, no sirve como negativo.
    """
    cont = cont if cont is not None else collections.Counter()
    jitter = rng.uniform(0.85, 1.15) if split == "train" else 1.0
    s = min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM * jitter)
    img = cv2.resize(lienzo, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else lienzo
    val = cv2.resize(valido, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    # Si la foto escalada es más chica que un tile (p. ej. a 10 mm/px) se completa ANTES de simular el
    # sensor: así el relleno también recibe blur, viñeteo y ruido, y no queda un escalón plano artificial.
    ph, pw = max(0, TILE_H - img.shape[0]), max(0, TILE_W - img.shape[1])
    if ph or pw:
        media = [float(v) for v in img.reshape(-1, img.shape[2]).mean(axis=0)]
        img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=media)
        val = cv2.copyMakeBorder(val, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)
    if MODO_COLOR == "gris":
        f = a_gris(img, rng, aleatorio=(split == "train"))
    else:
        f = img.astype(np.float32)
    img = degradar(f, rng, fuerte=(split == "train"))
    H, W = img.shape[:2]
    salida = []
    for y0 in posiciones(H, TILE_H):
        for x0 in posiciones(W, TILE_W):
            labs, danio_sin_etiqueta = [], False
            for k, bx1, by1, bx2, by2 in cajas:
                x1, y1, x2, y2 = bx1 * s - x0, by1 * s - y0, bx2 * s - x0, by2 * s - y0
                cx1, cy1, cx2, cy2 = max(x1, 0), max(y1, 0), min(x2, TILE_W), min(y2, TILE_H)
                if cx2 <= cx1 or cy2 <= cy1:
                    continue                          # la caja no toca este tile
                vw, vh = cx2 - cx1, cy2 - cy1
                # Fracción visible respecto de lo máximo que la caja puede ocupar en un tile: así una mancha
                # más grande que el tile (unida por unir_cortes) no desaparece de todos los tiles.
                area_ref = max(min(x2 - x1, TILE_W) * min(y2 - y1, TILE_H), 1e-6)
                if vw * vh / area_ref < MIN_VISIBLE or max(vw, vh) < MIN_BOX_PX or min(vw, vh) < 1:
                    danio_sin_etiqueta |= (k == DANIO_ID)
                    continue
                if SOLO_DANIO and k != DANIO_ID:
                    continue                          # en modo solo-daño, dirt es fondo
                labs.append((k, cx1, cy1, cx2, cy2))
            if not labs:
                if danio_sin_etiqueta:
                    cont["descartados_danio_sin_etiqueta"] += 1
                    continue
                if val[y0:y0 + TILE_H, x0:x0 + TILE_W].mean() < MIN_VALIDO:
                    cont["descartados_poca_pala"] += 1
                    continue
            salida.append((img[y0:y0 + TILE_H, x0:x0 + TILE_W].copy(), labs, x0, y0))
    return salida

def preparar_wtbs(img, split, rng):
    """WTBs2025: GSD desconocido y variable -> no se re-muestrea ni se corta en tiles; solo gris + sensor."""
    f = a_gris(img, rng, aleatorio=(split == "train")) if MODO_COLOR == "gris" else img.astype(np.float32)
    return degradar(f, rng, fuerte=(split == "train"))

In [ ]:
# ============================================================
# 5. Generar el dataset simulado
# ============================================================
if OUT_DS.exists():
    shutil.rmtree(OUT_DS)
negativos = collections.defaultdict(list)
stats = collections.defaultdict(collections.Counter)
lados_danio = collections.defaultdict(list)
t0 = time.time()

def guardar_labels(ruta, labs, w, h, split):
    with open(ruta, "w") as fh:
        for k, x1, y1, x2, y2 in labs:
            fh.write(f"{k} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}\n")
            stats[split][NOMBRES[k]] += 1
            if k == DANIO_ID:
                lados_danio[split].append(max(x2 - x1, y2 - y1))

def crear_split(split):
    (OUT_DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DS / "labels" / split).mkdir(parents=True, exist_ok=True)

# ---- 5a. DTU: foto reconstruida -> GSD objetivo -> tiles 640x400 ----
for split, lista in SPLITS.items():
    crear_split(split)
    n_var = VARIANTES_TRAIN if split == "train" else 1
    for i, foto in enumerate(lista):
        lienzo, valido, cajas = armar_foto(foto)
        for v in range(n_var):
            rng = np.random.default_rng(zlib.crc32(f"{SEED}-{foto}-{split}-{v}".encode()))
            for tile, labs, x0, y0 in generar_tiles(lienzo, valido, cajas, split, v, rng, stats[split]):
                nombre = f"{foto}_v{v}_{x0}_{y0}"
                cv2.imwrite(str(OUT_DS / "images" / split / f"{nombre}.png"), tile)
                guardar_labels(OUT_DS / "labels" / split / f"{nombre}.txt", labs, TILE_W, TILE_H, split)
                stats[split]["tiles"] += 1
                stats[split]["tiles_dtu"] += 1
                if not labs:
                    negativos[split].append(nombre)
        if i % 50 == 0:
            print(f"  DTU {split}: {i}/{len(lista)} fotos  ({time.time() - t0:.0f} s)")

# ---- 5b. WTBs2025_2cls: imagen completa, solo gris + sensor. Su test va aparte (test_wtbs) ----
WTBS_CLASE = collections.defaultdict(dict)   # split -> {archivo generado: clase original}
if WTBS:
    for sp, (im_dir, lb_dir) in WTBS.items():
        split = "test_wtbs" if sp == "test" else sp
        crear_split(split)
        n_var = VARIANTES_WTBS if split == "train" else 1
        rutas = sorted(p for p in im_dir.iterdir() if p.suffix.lower() in EXT_IMG)
        for p in rutas:
            lbl = lb_dir / f"{p.stem}.txt"
            filas = [l.split() for l in lbl.read_text().splitlines()] if lbl.exists() else []
            filas = [f for f in filas if len(f) >= 5]
            if not filas:                              # WTBs no trae pala limpia: sin etiqueta = error de origen
                stats[split]["wtbs_sin_etiqueta"] += 1
                continue
            img = cv2.imread(str(p))
            if img is None:
                stats[split]["wtbs_ilegibles"] += 1
                continue
            h, w = img.shape[:2]
            labs = []
            for f in filas:
                k, (cx, cy, bw, bh) = int(f[0]), map(float, f[1:5])
                if k != 0:                             # en WTBs2025_2cls todo es damage (0)
                    stats[split]["wtbs_clase_inesperada"] += 1
                    continue
                labs.append((DANIO_ID, max(cx - bw / 2, 0) * w, max(cy - bh / 2, 0) * h,
                             min(cx + bw / 2, 1) * w, min(cy + bh / 2, 1) * h))
            if not labs:
                continue
            for v in range(n_var):
                rng = np.random.default_rng(zlib.crc32(f"{SEED}-wtbs-{p.stem}-{split}-{v}".encode()))
                nombre = f"WTBS_{p.stem}_v{v}"
                cv2.imwrite(str(OUT_DS / "images" / split / f"{nombre}.png"), preparar_wtbs(img, split, rng))
                guardar_labels(OUT_DS / "labels" / split / f"{nombre}.txt", labs, w, h, split)
                stats[split]["tiles"] += 1
                stats[split]["imgs_wtbs"] += 1
                WTBS_CLASE[split][nombre] = WTBS_CLASE_ORIG.get(p.stem, "sin dato")
        print(f"  WTBs {sp} -> {split}: {stats[split]['imgs_wtbs']} imágenes  ({time.time() - t0:.0f} s)")

# ---- Negativos (solo DTU): en train se limitan a NEG_FRAC del split; en val y test se dejan todos ----
for split in SPLITS:
    pos = stats[split]["tiles"] - len(negativos[split])
    if split in NEG_LIMITAR:
        max_neg = int(NEG_FRAC / (1 - NEG_FRAC) * pos)
        sobran = negativos[split][:]
        random.Random(SEED).shuffle(sobran)
        for nombre in sobran[max_neg:]:
            (OUT_DS / "images" / split / f"{nombre}.png").unlink()
            (OUT_DS / "labels" / split / f"{nombre}.txt").unlink()
        negativos[split] = sobran[:max_neg]
    stats[split]["negativos"] = len(negativos[split])
    stats[split]["tiles"] = pos + stats[split]["negativos"]

print(f"\nDataset generado en {time.time() - t0:.0f} s -> {OUT_DS}")
for split in [s for s in ("train", "val", "test", "test_wtbs") if s in stats]:
    l = np.array(lados_danio[split]) if lados_danio[split] else np.array([0])
    print(f"{split:9s}: {dict(stats[split])}  |  lado mayor de 'damage' (px): "
          f"p10={np.percentile(l, 10):.0f}  mediana={np.median(l):.0f}  p90={np.percentile(l, 90):.0f}  "
          f"<8px: {np.mean(l < 8) * 100:.0f} %")

YAML = WORK / f"{OUT_DS.name}.yaml"
with open(YAML, "w") as fh:
    yaml.dump({"path": str(OUT_DS), "train": "images/train", "val": "images/val", "test": "images/test",
               "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
YAML_WTBS = None                     # el test de WTBs se evalúa con otro YAML (no entra al test de decisión)
if stats["test_wtbs"]["tiles"]:
    YAML_WTBS = WORK / f"{OUT_DS.name}_wtbs.yaml"
    with open(YAML_WTBS, "w") as fh:
        yaml.dump({"path": str(OUT_DS), "train": "images/train", "val": "images/val", "test": "images/test_wtbs",
                   "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
with open(WORK / f"{OUT_DS.name}_splits.json", "w") as fh:
    json.dump(SPLITS, fh, indent=1)
print("YAML:", YAML, "| YAML test WTBs:", YAML_WTBS)

In [ ]:
# ============================================================
# 6. Control visual: ¿la foto se reconstruyó bien? ¿los tiles tienen sentido?
# ============================================================
import matplotlib.pyplot as plt

COLORES = {0: (0, 0, 255), 1: (0, 200, 255)}   # 0 = damage (rojo), 1 = dirt (naranja), en BGR

foto_ej = sorted(con_danio)[0]
lienzo, valido, cajas = armar_foto(foto_ej)
vista = lienzo.copy()
for k, x1, y1, x2, y2 in cajas:
    cv2.rectangle(vista, (int(x1), int(y1)), (int(x2), int(y2)), COLORES.get(k, (255, 0, 0)), 6)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(vista, cv2.COLOR_BGR2RGB))
plt.title(f"{foto_ej} reconstruida: si los tiles no encajan, la grilla fila/columna está invertida")
plt.axis("off"); plt.show()

ejemplos = sorted((OUT_DS / "labels" / "train").glob("*.txt"))
ejemplos = [e for e in ejemplos if e.stat().st_size > 0]
random.Random(1).shuffle(ejemplos)
fig, ax = plt.subplots(3, 4, figsize=(20, 10))
for a, lbl in zip(ax.flat, ejemplos[:12]):
    im = cv2.imread(str(OUT_DS / "images" / "train" / f"{lbl.stem}.png"))
    for line in lbl.read_text().splitlines():
        k, cx, cy, bw, bh = line.split()
        k, cx, cy, bw, bh = int(k), float(cx) * TILE_W, float(cy) * TILE_H, float(bw) * TILE_W, float(bh) * TILE_H
        cv2.rectangle(im, (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2)),
                      COLORES.get(k, (255, 0, 0)), 2)
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(lbl.stem, fontsize=8); a.axis("off")
plt.suptitle(f"Tiles de train simulando el OV9281 a {GSD_OBJETIVO_MM:g} mm/px (rojo = damage, naranja = dirt)")
plt.tight_layout(); plt.savefig(WORK / "preview_tiles.jpg", dpi=80); plt.show()

In [ ]:
# ============================================================
# 7. Entrenamiento
# ============================================================
import torch
from ultralytics import YOLO

N_GPU = torch.cuda.device_count()
DEVICE = 0 if N_GPU >= 1 else "cpu"   # con 2xT4 se puede probar [0, 1] (multi-GPU desde notebook a veces falla)
n_train = len(list((OUT_DS / "images" / "train").glob("*.png")))
CACHE = "ram" if n_train < 6000 else False
print(f"GPUs: {N_GPU}  device={DEVICE}  tiles de train: {n_train}  cache={CACHE}")

model = YOLO(MODELO_BASE)
model.train(
    data=str(YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
    device=DEVICE, fraction=FRACTION, workers=4, cache=CACHE,
    optimizer="auto", cos_lr=True, close_mosaic=15,
    seed=SEED, deterministic=True,
    # Aumentos: sin color en gris (no hay tono ni saturación); brillo sí
    hsv_h=0.0, hsv_s=0.0 if MODO_COLOR == "gris" else 0.5, hsv_v=0.4,
    # Geometría: la pala aparece en cualquier ángulo; escala acotada (el GSD lo fija la distancia de vuelo)
    degrees=15, translate=0.1, scale=0.3, perspective=0.0005,
    fliplr=0.5, flipud=0.5, mosaic=1.0, mixup=0.0,
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True, plots=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
print("Mejor modelo:", BEST)

In [ ]:
# ============================================================
# 8. Evaluación: métricas estándar en TEST + métricas de inspección (umbral elegido en VAL)
# ============================================================
def resumen_val(m, titulo):
    filas = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        filas.append((NOMBRES[int(c)], p, r, ap50, ap))
    print(f"\n{titulo}")
    print(f"{'clase':8s} {'P':>6s} {'R':>6s} {'mAP50':>6s} {'mAP50-95':>8s}")
    for n, p, r, a50, a in filas:
        print(f"{n:8s} {p:6.3f} {r:6.3f} {a50:6.3f} {a:8.3f}")
    return {n: {"P": p, "R": r, "mAP50": a50, "mAP50-95": a} for n, p, r, a50, a in filas}

best = YOLO(str(BEST))
m_test = best.val(data=str(YAML), split="test", imgsz=IMGSZ, batch=16, device=DEVICE,
                  project=str(RUN_DIR), name="test_fp32", exist_ok=True)
res_fp32 = resumen_val(m_test, "TEST (DTU, decide) — modelo float (best.pt)")
res_wtbs_fp32 = None
if YAML_WTBS:
    m_w = best.val(data=str(YAML_WTBS), split="test", imgsz=IMGSZ, batch=16, device=DEVICE,
                   project=str(RUN_DIR), name="test_wtbs_fp32", exist_ok=True)
    res_wtbs_fp32 = resumen_val(m_w, "TEST WTBs2025 (referencia, no decide) — modelo float")

def iou(a, b):
    """a: (N,4), b: (M,4) en xyxy -> (N,M)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def emparejar(pr, cf, gt, iou_min):
    """Asignación 1 a 1 por confianza: cada predicción acierta como mucho una caja real.
    Devuelve (índices de las cajas reales detectadas, cantidad de falsos positivos)."""
    m = iou(pr, gt)
    usados, fp = set(), 0
    for j in np.argsort(-cf):
        cand = [g for g in np.argsort(-m[j]) if m[j, g] >= iou_min and g not in usados] if len(gt) else []
        if cand:
            usados.add(cand[0])
        else:
            fp += 1
    return usados, fp

UMBRALES = np.round(np.arange(0.05, 0.95, 0.05), 2)

def cubo(lado):
    return "<8 px" if lado < 8 else "8-16 px" if lado < 16 else "16-32 px" if lado < 32 else ">32 px"

def evaluar_inspeccion(modelo, split="test", umbral=None, prefijo=None, clase_de=None,
                       iou_min=0.3, device=DEVICE, batch=16):
    """Recall de damage (total, por tamaño y, si se pasa clase_de, por clase original), precisión y falsos
    positivos cada 100 tiles limpios. Con umbral=None el umbral se ELIGE en este split (máximo recall con
    P >= P_MIN); si se pasa, se usa fijo. Así el umbral sale de val y test se mide sin fuga."""
    rutas = sorted(p for p in (OUT_DS / "images" / split).glob("*.png") if not prefijo or p.name.startswith(prefijo))
    datos = []   # (stem, gt damage (N,4), pred damage (M,4), conf (M,), es_negativo)
    for i in range(0, len(rutas), batch):
        lote = rutas[i:i + batch]
        res = modelo.predict([str(p) for p in lote], imgsz=IMGSZ, conf=0.01, verbose=False, device=device)
        for p, r in zip(lote, res):
            gt, todas = [], 0
            for line in (OUT_DS / "labels" / split / f"{p.stem}.txt").read_text().splitlines():
                k, cx, cy, bw, bh = line.split(); todas += 1
                if int(k) == DANIO_ID:
                    h, w = r.orig_shape
                    cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
                    gt.append([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2])
            sel = r.boxes.cls.cpu().numpy().astype(int) == DANIO_ID
            datos.append((p.stem, np.array(gt).reshape(-1, 4), r.boxes.xyxy.cpu().numpy()[sel],
                          r.boxes.conf.cpu().numpy()[sel], todas == 0))
    n_neg = int(sum(d[4] for d in datos))

    def medir(t):
        tp = fp = fn = fp_neg = 0
        detectados = []
        for _, gt, pr, cf, neg in datos:
            s = cf >= t
            usados, f = emparejar(pr[s], cf[s], gt, iou_min)
            tp += len(usados); fp += f; fp_neg += f * neg; fn += len(gt) - len(usados)
            detectados.append(usados)
        P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
        return {"umbral": float(t), "P": P, "R": R, "F1": 2 * P * R / max(P + R, 1e-9),
                "FP_cada_100_limpios": 100 * fp_neg / n_neg if n_neg else None}, detectados

    tabla = [medir(t)[0] for t in UMBRALES]
    if umbral is None:
        validos = [f for f in tabla if f["P"] >= P_MIN]
        umbral = (max(validos, key=lambda f: f["R"]) if validos else max(tabla, key=lambda f: f["F1"]))["umbral"]
        origen = f"elegido en {split}" + (f" ({prefijo}*)" if prefijo else "")
    else:
        origen = "fijo (elegido en val)"
    elegido, detectados = medir(umbral)

    # Recall por tamaño (y por clase original) con el MISMO emparejamiento 1 a 1 del umbral elegido
    cubos = {c: [0, 0] for c in ("<8 px", "8-16 px", "16-32 px", ">32 px")}
    por_clase = collections.defaultdict(lambda: [0, 0])
    for (stem, gt, _, _, _), usados in zip(datos, detectados):
        for g in range(len(gt)):
            c = cubos[cubo(max(gt[g, 2] - gt[g, 0], gt[g, 3] - gt[g, 1]))]
            c[1] += 1; c[0] += int(g in usados)
            if clase_de is not None:
                pc = por_clase[clase_de.get(stem, "sin dato")]
                pc[1] += 1; pc[0] += int(g in usados)

    fp_txt = lambda v: "n/a" if v is None else f"{v:.1f}"
    print(f"\nInspección ({split}{', ' + prefijo + '*' if prefijo else ''}, IoU >= {iou_min}, "
          f"{len(datos)} imágenes, {n_neg} limpias)")
    print(f"{'umbral':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'FP/100 limpios':>15s}")
    for f in tabla[::2]:
        print(f"{f['umbral']:6.2f} {f['P']:6.3f} {f['R']:6.3f} {f['F1']:6.3f} {fp_txt(f['FP_cada_100_limpios']):>15s}")
    print(f"\nUmbral {umbral:.2f} [{origen}]  ->  P={elegido['P']:.3f}  R={elegido['R']:.3f}  "
          f"FP/100 limpios={fp_txt(elegido['FP_cada_100_limpios'])}")
    print("Recall de damage por tamaño:", {k: f"{v[0] / max(v[1], 1):.2f} (n={v[1]})" for k, v in cubos.items()})
    if clase_de is not None:
        print("Recall por clase original:",
              {k: f"{v[0] / max(v[1], 1):.2f} (n={v[1]})" for k, v in sorted(por_clase.items())})
    return {"split": split, "prefijo": prefijo, "umbral": float(umbral), "origen_umbral": origen,
            "n_imagenes": len(datos), "n_limpias": n_neg, "tabla": tabla, "elegido": elegido,
            "recall_por_tamano": {k: {"recall": v[0] / max(v[1], 1), "n": v[1]} for k, v in cubos.items()},
            "recall_por_clase_original": ({k: {"recall": v[0] / max(v[1], 1), "n": v[1]}
                                           for k, v in por_clase.items()} if clase_de is not None else None)}

# El umbral se elige en val (solo la parte DTU: mismo dominio que el test de decisión) y se aplica fijo en test.
insp_val_fp32 = evaluar_inspeccion(best, split="val", prefijo="DJI_")
insp_fp32 = evaluar_inspeccion(best, split="test", umbral=insp_val_fp32["umbral"])
insp_wtbs_fp32 = (evaluar_inspeccion(best, split="test_wtbs", umbral=insp_val_fp32["umbral"],
                                     clase_de=WTBS_CLASE["test_wtbs"]) if YAML_WTBS else None)

In [ ]:
# ============================================================
# 9. Export a Edge TPU (int8) calibrado con tiles propios + validación del modelo cuantizado
# ============================================================
# Se exporta a 640x640 para poder validar el .tflite con Ultralytics; la entrada rectangular
# (416x640, ~35 % más rápida) queda como optimización posterior.

def instalar_edgetpu_compiler():
    """Ultralytics instala el compilador con apt-key, que falla en Ubuntu reciente. Se instala antes con el
    método actual (keyring firmado). Si igual falla, el export deja el .tflite int8 y solo falta compilarlo."""
    if shutil.which("edgetpu_compiler"):
        return True
    sudo = "" if os.geteuid() == 0 else "sudo "
    pasos = [
        f"curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | {sudo}gpg --batch --yes --dearmor "
        f"-o /usr/share/keyrings/coral-edgetpu.gpg",
        f'echo "deb [signed-by=/usr/share/keyrings/coral-edgetpu.gpg] https://packages.cloud.google.com/apt '
        f'coral-edgetpu-stable main" | {sudo}tee /etc/apt/sources.list.d/coral-edgetpu.list',
        f"{sudo}apt-get update -qq",
        f"{sudo}apt-get install -y -qq edgetpu-compiler",
    ]
    for p in pasos:
        r = subprocess.run(p, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print("No se pudo instalar edgetpu_compiler:", (r.stderr or r.stdout)[-500:])
            return False
    return shutil.which("edgetpu_compiler") is not None

print("edgetpu_compiler disponible:", instalar_edgetpu_compiler())

# Calibración int8 con una muestra ALEATORIA de val (DTU + WTBs, con y sin daño). Con fraction= Ultralytics
# toma las primeras imágenes en orden alfabético (pocas fotos DTU y nada de WTBs), por eso se arma aparte.
for sub in ("images", "labels"):
    shutil.rmtree(OUT_DS / sub / "calib", ignore_errors=True)
    (OUT_DS / sub / "calib").mkdir(parents=True)
cal = sorted((OUT_DS / "images" / "val").glob("*.png"))
random.Random(SEED).shuffle(cal)
for p in cal[:N_CALIB]:
    shutil.copy(p, OUT_DS / "images" / "calib" / p.name)
    shutil.copy(OUT_DS / "labels" / "val" / f"{p.stem}.txt", OUT_DS / "labels" / "calib" / f"{p.stem}.txt")
YAML_CALIB = WORK / f"{OUT_DS.name}_calib.yaml"
with open(YAML_CALIB, "w") as fh:
    yaml.dump({"path": str(OUT_DS), "train": "images/calib", "val": "images/calib",
               "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
print(f"Calibración int8: {min(N_CALIB, len(cal))} imágenes de val al azar")

try:
    salida_export = best.export(format="edgetpu", imgsz=IMGSZ, data=str(YAML_CALIB))
    print("Export:", salida_export)
except Exception as e:
    print("El export a Edge TPU falló:", repr(e))

WDIR = BEST.parent
TFLITE_INT8 = sorted(p for p in WDIR.rglob("*full_integer_quant.tflite") if "edgetpu" not in p.name)
TFLITE_TPU = sorted(WDIR.rglob("*_edgetpu.tflite"))
print("int8 (CPU):", TFLITE_INT8)
print("Edge TPU  :", TFLITE_TPU)
if TFLITE_INT8 and not TFLITE_TPU:
    print("Hay .tflite int8 pero no se compiló para la TPU: compilarlo en la PC con "
          "'edgetpu_compiler <archivo>_full_integer_quant.tflite'.")

# Log del compilador: cuántas operaciones corren en la TPU y cuántas caen en la CPU
for log in sorted(WDIR.rglob("*edgetpu*.log")):
    texto = log.read_text(errors="ignore")
    print(f"\n--- {log.name} ---")
    print("\n".join(l for l in texto.splitlines() if "Number of operations" in l or "Edge TPU" in l[:40]) or texto[-1500:])

res_int8 = insp_val_int8 = insp_int8 = insp_wtbs_int8 = None
if TFLITE_INT8:
    q = YOLO(str(TFLITE_INT8[0]), task="detect")
    m_q = q.val(data=str(YAML), split="test", imgsz=IMGSZ, batch=1, device="cpu",
                project=str(RUN_DIR), name="test_int8", exist_ok=True)
    res_int8 = resumen_val(m_q, "TEST (DTU) — modelo int8 (el que corre en la Coral)")
    d = NOMBRES[DANIO_ID]
    print(f"\nCaída por cuantización en mAP50 de damage: "
          f"{(res_fp32[d]['mAP50'] - res_int8[d]['mAP50']) * 100:.1f} puntos (objetivo: <= 3)")
    # El umbral para la Coral se elige con el propio modelo int8, también en val
    insp_val_int8 = evaluar_inspeccion(q, split="val", prefijo="DJI_", device="cpu", batch=1)
    insp_int8 = evaluar_inspeccion(q, split="test", umbral=insp_val_int8["umbral"], device="cpu", batch=1)
    if YAML_WTBS:
        insp_wtbs_int8 = evaluar_inspeccion(q, split="test_wtbs", umbral=insp_val_int8["umbral"],
                                            clase_de=WTBS_CLASE["test_wtbs"], device="cpu", batch=1)
else:
    print("No hay modelo int8: revisar el error del export.")

In [ ]:
# ============================================================
# 10. Entregables para descargar
# ============================================================
ENT = WORK / "entregables" / RUN_NAME
ENT.mkdir(parents=True, exist_ok=True)
copiar = [BEST, YAML, WORK / f"{OUT_DS.name}_splits.json", WORK / "preview_tiles.jpg",
          RUN_DIR / "results.csv", RUN_DIR / "results.png"]
copiar += list(RUN_DIR.glob("*PR_curve.png")) + list(RUN_DIR.glob("confusion_matrix*.png"))
copiar += TFLITE_TPU + TFLITE_INT8 + list(WDIR.rglob("*edgetpu*.log"))
for p in copiar:
    if Path(p).exists():
        shutil.copy(p, ENT / Path(p).name)

config = {k: globals()[k] for k in ["SEED", "USAR_WTBS", "VARIANTES_WTBS", "GSD_ORIG_MM", "GSD_OBJETIVO_MM",
                                    "MODO_COLOR", "SOLO_DANIO", "NOMBRES", "TILE_W", "TILE_H", "SOLAPE",
                                    "VARIANTES_TRAIN", "NEG_FRAC", "NEG_LIMITAR", "MIN_BOX_PX", "MIN_VISIBLE",
                                    "MIN_VALIDO", "MODELO_BASE", "EPOCHS", "PATIENCE", "BATCH", "IMGSZ",
                                    "FRACTION", "P_MIN", "N_CALIB"]}
umbral_coral = (insp_val_int8 or insp_val_fp32)["umbral"]
with open(ENT / "metricas.json", "w") as fh:
    json.dump({"config": config,
               "umbral_sugerido": {"int8_coral": insp_val_int8["umbral"] if insp_val_int8 else None,
                                   "fp32": insp_val_fp32["umbral"], "elegido_en": "val (solo DTU)"},
               "test_fp32": res_fp32, "test_int8": res_int8, "test_wtbs_fp32": res_wtbs_fp32,
               "inspeccion_val_fp32": insp_val_fp32, "inspeccion_fp32": insp_fp32,
               "inspeccion_val_int8": insp_val_int8, "inspeccion_int8": insp_int8,
               "inspeccion_wtbs_fp32": insp_wtbs_fp32, "inspeccion_wtbs_int8": insp_wtbs_int8,
               "dataset": {k: dict(v) for k, v in stats.items()}}, fh, indent=1, default=float)

archivo = shutil.make_archive(str(WORK / "entregables"), "zip", WORK / "entregables")
print("Listo:", archivo)
print("Contenido:", sorted(p.name for p in ENT.iterdir()))
print(f"\nUmbral sugerido para la Coral: {umbral_coral:.2f} (usar como CONF en 'Prueba modelo.ipynb').")
print("Para probar en la PC: copiar best.pt a 'Proyecto final/Modelo/Modelo nuevo/best.pt' y usar MODELO = 'nuevo'.")